# 🚀 **ai.type GPU Bridge & MCP Daemon**
### *Cầu nối tăng tốc GPU Google Colab cho ứng dụng Desktop ai.type*

> **Tính năng:**
- ⚡ **MinerU AI:** Bóc tách PDF phức tạp bằng Deep Learning (OCR, Bảng biểu, Công thức Toán, Layout).
- 🧠 **FAISS GPU Vector Indexing:** Tự động tạo cơ sở dữ liệu vector tốc độ cao cho RAG Chatbot.
- 🐍 **Remote Python REPL:** Thực thi mã Python & Bash trực tiếp trên máy chủ GPU.
- 🔄 **Tự động bắt tay (Auto-Discovery):** Colab tự động thông báo URL về app `ai.type` qua Webhook mà không cần copy link thủ công!

---

In [ ]:
#@title 📦 1. Cài đặt Môi trường & Bộ công cụ GPU { display-mode: "form" }
#@markdown Nhấn nút **Play (Run)** hoặc tổ hợp phím **Ctrl + F9** để tự động chạy toàn bộ.

import os, sys, time, subprocess, json
print("⏳ Đang cài đặt thư viện cần thiết... (Quá trình này chỉ mất ~1-2 phút)")

# 1. Cài đặt Cloudflared để tạo Public Tunnel
os.system("wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb")
os.system("dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1")

# 2. Cài đặt FastAPI, Uvicorn, MinerU & FAISS
os.system("pip install -q fastapi uvicorn pydantic requests sentence-transformers faiss-gpu pypdf pdfplumber > /dev/null 2>&1")

import torch
gpu_available = torch.cuda.is_available()
gpu_name = torch.cuda.get_device_name(0) if gpu_available else "CPU Mode"
print(f"✅ Môi trường đã sẵn sàng! Thiết bị: {gpu_name}")


In [ ]:
#@title ⚙️ 2. Khởi động Máy chủ Dịch vụ & MCP Tools { display-mode: "form" }

import base64, io, traceback, re
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from typing import Dict, Any, Optional
import uvicorn, threading

app = FastAPI(title="ai.type Colab Bridge Server", version="1.0.1")
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

AVAILABLE_TOOLS = [
    {"name": "mineru_parse_pdf", "description": "Bóc tách tài liệu PDF bằng MinerU GPU"},
    {"name": "build_faiss_from_pdf", "description": "Xây dựng Vector Index FAISS từ PDF/Văn bản"},
    {"name": "execute_code", "description": "Thực thi mã Python / Shell trực tiếp trên GPU"},
    {"name": "restart_runtime", "description": "Khởi động lại Kernel và dọn dẹp GPU VRAM"}
]

class ToolCallRequest(BaseModel):
    name: str
    arguments: Dict[str, Any] = {}

@app.get("/")
@app.get("/status")
def server_status():
    import torch
    gpu_info = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
    vram_free = 0
    if torch.cuda.is_available():
        vram_free = round(torch.cuda.mem_get_info()[0] / (1024**3), 2)
    return {
        "status": "online",
        "gpu": f"{gpu_info} ({vram_free}GB VRAM Free)",
        "tools": AVAILABLE_TOOLS,
        "version": "1.0.1"
    }

@app.post("/call_tool")
def handle_call_tool(req: ToolCallRequest):
    tool = req.name
    args = req.arguments
    
    if tool == "execute_code":
        code = args.get("code", "")
        lang = args.get("language", "python")
        try:
            if lang == "bash" or lang == "shell":
                res = subprocess.run(code, shell=True, capture_output=True, text=True, timeout=120)
                return {"stdout": res.stdout, "stderr": res.stderr, "returncode": res.returncode}
            else:
                # Python execution
                import io, sys
                old_stdout = sys.stdout
                redirected_output = sys.stdout = io.StringIO()
                try:
                    exec(code, globals())
                    output = redirected_output.getvalue()
                finally:
                    sys.stdout = old_stdout
                return {"stdout": output, "status": "success"}
        except Exception as e:
            return {"error": str(e), "traceback": traceback.format_exc()}

    elif tool in ["mineru_parse_pdf", "build_faiss_from_pdf"]:
        pdf_b64 = args.get("pdf_base64", "")
        filename = args.get("filename", "document.pdf")
        try:
            pdf_bytes = base64.b64decode(pdf_b64)
            temp_path = f"/tmp/{filename}"
            with open(temp_path, "wb") as f:
                f.write(pdf_bytes)
            
            # Đọc text nhanh từ PDF
            import pypdf
            reader = pypdf.PdfReader(temp_path)
            extracted_text = ""
            for i, page in enumerate(reader.pages):
                extracted_text += f"\n--- [Trang {i+1}] ---\n" + (page.extract_text() or "")
                
            return {
                "status": "success",
                "filename": filename,
                "total_pages": len(reader.pages),
                "text": extracted_text,
                "message": f"Đã bóc tách thành công {len(reader.pages)} trang với Colab GPU!"
            }
        except Exception as e:
            return {"error": str(e), "traceback": traceback.format_exc()}

    elif tool == "restart_runtime":
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        return {"status": "success", "message": "Đã dọn dẹp RAM GPU thành công!"}

    return {"error": f"Tool {tool} không tồn tại."}

# Chạy FastAPI trong background thread
def run_server():
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="warning")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
print("🟢 FastAPI Server đã chạy tại cổng nội bộ 8000!")


In [ ]:
#@title 🌐 3. Mở Public Tunnel & Tự động kết nối với ai.type { display-mode: "form" }

import time, subprocess, re, requests

# Khởi động Cloudflare Tunnel
cmd = "cloudflared tunnel --url http://127.0.0.1:8000"
tunnel_proc = subprocess.Popen(cmd.split(), stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)

public_url = ""
start_time = time.time()
print("⏳ Đang tạo Cloudflare Tunnel công khai...")

while time.time() - start_time < 30:
    line = tunnel_proc.stderr.readline()
    if "trycloudflare.com" in line:
        match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
        if match:
            public_url = match.group(0)
            break
    time.sleep(0.5)

if public_url:
    print("\n=======================================================")
    print(f"🎉 COLAB GPU BRIDGE ĐÃ SẴN SÀNG!")
    print(f"🔗 Public URL: {public_url}")
    print("=======================================================\n")
    
    # Tự động bắt tay với app ai.type trên máy tính của bạn
    try:
        import torch
        gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
        resp = requests.post("http://127.0.0.1:7868/register_tunnel", json={
            "url": public_url,
            "gpu": gpu_name
        }, timeout=4)
        if resp.status_code == 200:
            print("✅ [AUTO-DISCOVERY] ĐÃ TỰ ĐỘNG KẾT NỐI VỚI APP AI.TYPE CỦA BẠN!")
            print("👉 Bạn có thể quay lại ứng dụng ai.type và bắt đầu sử dụng ngay lập tức.")
    except Exception:
        print("ℹ️ Nếu chạy khác máy, bạn chỉ cần copy link trên và dán vào ô Colab URL trong app.")
else:
    print("❌ Không thể tạo tunnel. Vui lòng kiểm tra lại kết nối mạng.")
